In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib

# add path to project root

import sys
PROJECT_ROOT = Path.cwd()

if "notebooks" in PROJECT_ROOT.parts:
    PROJECT_ROOT = PROJECT_ROOT.parent
    
sys.path.append(str(PROJECT_ROOT))

    
from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble.ensemble import Ensemble
from src.model.game_model_collection import GamingModelCollection
from src.model.social_media_collection import SocialMediaModelCollection
from src.model.bert_collection import BertToxicityModelCollection
from src.ensemble.score import ClassificationMetrics


CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}


print(PROJECT_ROOT)

# social media
social_paths = list((CONFIG['model_dir'] / 'binary' ).glob('social_media_*.joblib'))

scaler_path = CONFIG['model_dir'] / 'helper' / 'social_media_max_abs_scaler.joblib'
nb_tfidf_path = CONFIG['model_dir'] / 'helper' / 'social_media_nb_tfidf.joblib'


# gaming
model_dir = CONFIG['model_dir'] / "binary"
gaming_model_paths = list(model_dir.glob("gaming_all_*.joblib"))



c:\Users\nyuss\OneDrive\Documentos\Bars\Portfollio\Portfollio\Projects\Project-GamingToxicityDetection


In [2]:
np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

# Load Data

In [3]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"


# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

In [4]:
wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

# combine holdout datasets
val = pd.concat([wot_val, dota_val], ignore_index=True)

# Prepare Objects

In [5]:
# instantiate preprocessor and labeler
gaming_preprocessor = GamingTextPreprocessor()
gaming_labeler = GamingTextLabeler()
metrics = ClassificationMetrics()

gaming_preprocessor.fit(train, text_column=tc)

In [6]:
# convert dota labels 
train = gaming_labeler.convert_binary(
    train, 
    label_column=lc, 
    output_column='label_binary'
)
val = gaming_labeler.convert_binary(
    val, 
    label_column=lc, 
    output_column='label_binary'
)


In [7]:
gaming_model_collection = GamingModelCollection(
    model_joblibs = gaming_model_paths
)

In [8]:
gaming_model_collection.classifiers.keys()

dict_keys(['wot_Logistic_Regression', 'wot_LinearSVC', 'wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True)', 'wot_LinearSVC_+_Calibrated(isotonic,_ensemble=True)', 'dota_Logistic_Regression', 'dota_LinearSVC', 'dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False)', 'dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True)', 'combined_Logistic_Regression', 'combined_LinearSVC', 'combined_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True)', 'combined_LinearSVC_+_Calibrated(sigmoid,_ensemble=False)'])

In [9]:
social_media_collection = SocialMediaModelCollection(
    model_joblibs=social_paths,
    scaler_path=scaler_path,
    nb_tfidf_path=nb_tfidf_path,
)

In [10]:
social_media_collection.classifiers

{WindowsPath('c:/Users/nyuss/OneDrive/Documentos/Bars/Portfollio/Portfollio/Projects/Project-GamingToxicityDetection/models/binary/social_media_ComplementNB_model.joblib'): Pipeline(steps=[('clf', ComplementNB(alpha=1.2006196407511314))]),
 WindowsPath('c:/Users/nyuss/OneDrive/Documentos/Bars/Portfollio/Portfollio/Projects/Project-GamingToxicityDetection/models/binary/social_media_LinearSVC_(Optuna)_model.joblib'): Pipeline(steps=[('clf',
                  LinearSVC(C=0.7813882258601701, class_weight='balanced',
                            dual=False, max_iter=5000, penalty='l1',
                            random_state=42))]),
 WindowsPath('c:/Users/nyuss/OneDrive/Documentos/Bars/Portfollio/Portfollio/Projects/Project-GamingToxicityDetection/models/binary/social_media_LogisticRegressionCV_model.joblib'): Pipeline(steps=[('clf',
                  LogisticRegressionCV(class_weight='balanced',
                                       cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=

In [11]:
bert_binary_collection = BertToxicityModelCollection(
    model_names=["dota_binary", "jigsaw_binary", "wot_binary"]
)

Loading dota_binary from jforward/bert-toxicity/dota_binary_model...
Loading jigsaw_binary from jforward/bert-toxicity/jigsaw_binary_model...
Loading wot_binary from jforward/bert-toxicity/wot_binary_model...


In [12]:
social_gaming_ensemble = Ensemble(
    model_collections=[social_media_collection, gaming_model_collection, bert_binary_collection]
)

# Run Simple Ensemble 

In [13]:
pred_train = social_gaming_ensemble.predict_simple_majority(train[tc])
metrics.metrics(train['label_binary'], pred_train)

Predicting with SimpleMajority...
Input has 53707 samples.
social_media_ComplementNB_model: (53707,)
social_media_LinearSVC_(Optuna)_model: (53707,)
social_media_LogisticRegressionCV_model: (53707,)
wot_Logistic_Regression: (53707,)
wot_LinearSVC: (53707,)
wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True): (53707,)
wot_LinearSVC_+_Calibrated(isotonic,_ensemble=True): (53707,)
dota_Logistic_Regression: (53707,)
dota_LinearSVC: (53707,)
dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False): (53707,)
dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True): (53707,)
combined_Logistic_Regression: (53707,)
combined_LinearSVC: (53707,)
combined_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True): (53707,)
combined_LinearSVC_+_Calibrated(sigmoid,_ensemble=False): (53707,)
dota_binary: (53707,)
jigsaw_binary: (53707,)
wot_binary: (53707,)
Collected predictions from 18 models.
Prediction matrix shape: (53707, 18)
Aggregating predictions from 18 models...


{'CV Macro F1': 0.9026393662138283,
 'CV Weighted F1': 0.9321364523306177,
 'Accuracy': 0.9351667380415961,
 'Coverage': 1.0,
 'Precision': 0.9701598219704632,
 'FPR': 0.00720760341078453,
 'FNR': 0.24941305368602285,
 'TPR': 0.7505869463139772,
 'TNR': 0.9927923965892155}

In [14]:
pred_val = social_gaming_ensemble.predict_simple_majority(val[tc])
metrics.metrics(val['label_binary'], pred_val)

Predicting with SimpleMajority...
Input has 17056 samples.
social_media_ComplementNB_model: (17056,)
social_media_LinearSVC_(Optuna)_model: (17056,)
social_media_LogisticRegressionCV_model: (17056,)
wot_Logistic_Regression: (17056,)
wot_LinearSVC: (17056,)
wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True): (17056,)
wot_LinearSVC_+_Calibrated(isotonic,_ensemble=True): (17056,)
dota_Logistic_Regression: (17056,)
dota_LinearSVC: (17056,)
dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False): (17056,)
dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True): (17056,)
combined_Logistic_Regression: (17056,)
combined_LinearSVC: (17056,)
combined_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True): (17056,)
combined_LinearSVC_+_Calibrated(sigmoid,_ensemble=False): (17056,)
dota_binary: (17056,)
jigsaw_binary: (17056,)
wot_binary: (17056,)
Collected predictions from 18 models.
Prediction matrix shape: (17056, 18)
Aggregating predictions from 18 models...


{'CV Macro F1': 0.8198054924215161,
 'CV Weighted F1': 0.8831220604428022,
 'Accuracy': 0.8893644465290806,
 'Coverage': 1.0,
 'Precision': 0.8300653594771242,
 'FPR': 0.035053554040895815,
 'FNR': 0.38299595141700404,
 'TPR': 0.617004048582996,
 'TNR': 0.9649464459591042}

# Run Weighted Majority Ensemble 

In [16]:
weights = social_gaming_ensemble.fit_weighted_majority(
    X_val=train[tc],
    y_val=train['label_binary'],
    score_func=metrics.score,
    n_trials=500,
    random_state=CONFIG['seed']
)[0]

In [17]:
weights

array([0.05775814, 0.00236503, 0.04977899, 0.00473355, 0.10858845,
       0.02955511, 0.04662467, 0.03615981, 0.03636258, 0.07227485,
       0.01316597, 0.20768636, 0.02183354, 0.04309042, 0.0476378 ,
       0.05308232, 0.07417326, 0.09512916])

In [18]:
pred_train = social_gaming_ensemble.predict_weighted_majority(
    X=train[tc],
    weights=weights
)
metrics.metrics(train['label_binary'], pred_train)

{'CV Macro F1': 0.9682724687801103,
 'CV Weighted F1': 0.9769971116262947,
 'Accuracy': 0.9770048597017149,
 'Coverage': 1.0,
 'Precision': 0.9525601819179801,
 'FPR': 0.014781695130592,
 'FNR': 0.04930349037408045,
 'TPR': 0.9506965096259196,
 'TNR': 0.985218304869408}

In [19]:
pred_val = social_gaming_ensemble.predict_weighted_majority(
    X=val[tc],
    weights=weights
)
metrics.metrics(val['label_binary'], pred_val)

{'CV Macro F1': 0.8460383018697265,
 'CV Weighted F1': 0.8964717382715283,
 'Accuracy': 0.8976899624765479,
 'Coverage': 1.0,
 'Precision': 0.782258064516129,
 'FPR': 0.05662497191221631,
 'FNR': 0.2669365721997301,
 'TPR': 0.7330634278002699,
 'TNR': 0.9433750280877837}

# Run Weighted Confidence Majority Ensemble 

In [22]:
res = social_gaming_ensemble.fit_weighted_confidence_majority_optuna_cv(
    X = train[tc],
    y = train['label_binary'],
    score_func=metrics.score,
    n_trials=100,
    random_state=CONFIG['seed']
)

Constructing confidence tensor...


[I 2026-05-07 23:26:58,746] A new study created in memory with name: no-name-15df32ba-9c03-4324-8d2e-a29eb2100fe6
[I 2026-05-07 23:26:58,823] Trial 0 finished with value: 0.9190797463154519 and parameters: {'weight_0': 0.09002498359948537, 'weight_1': 0.24824433183856792, 'weight_2': 0.7099827053797744, 'weight_3': 0.11397955997969646, 'weight_4': 0.8752936353710526, 'weight_5': 0.05094125157566369, 'weight_6': 0.9079399348989289, 'weight_7': 0.7041352155447747, 'weight_8': 0.5736303169273469, 'weight_9': 0.5726684132685568, 'weight_10': 0.8316264150230945, 'weight_11': 0.7135711117868813, 'weight_12': 0.03657501623765234, 'weight_13': 0.43691084849471706, 'weight_14': 0.26593472891661585, 'weight_15': 0.07560124637707559, 'weight_16': 0.629294237460905, 'weight_17': 0.9329056977041611}. Best is trial 0 with value: 0.9190797463154519.
[I 2026-05-07 23:26:58,888] Trial 1 finished with value: 0.9182135603009215 and parameters: {'weight_0': 0.6173948594201659, 'weight_1': 0.00992314720583

Total models in ensemble: 18
Expected confidence shape: (n_samples=53707, n_classes=2)


[I 2026-05-07 23:26:58,967] Trial 2 finished with value: 0.8957989078067694 and parameters: {'weight_0': 0.3194338199463908, 'weight_1': 0.8078683144167786, 'weight_2': 0.6104134185476264, 'weight_3': 0.4474610094775442, 'weight_4': 0.6896621364522744, 'weight_5': 0.7121520873520022, 'weight_6': 0.9461798743187068, 'weight_7': 0.6941472629811276, 'weight_8': 0.2820268141569239, 'weight_9': 0.18779551395531047, 'weight_10': 0.025073773662082376, 'weight_11': 0.1923496480418878, 'weight_12': 0.18254314966918916, 'weight_13': 0.5900367911247051, 'weight_14': 0.41910223356860427, 'weight_15': 0.29588273805569376, 'weight_16': 0.5184943979543866, 'weight_17': 0.14134602050944284}. Best is trial 0 with value: 0.9190797463154519.
[I 2026-05-07 23:26:59,037] Trial 3 finished with value: 0.9044204995155909 and parameters: {'weight_0': 0.3579097256209228, 'weight_1': 0.26565022172873, 'weight_2': 0.5256286376443086, 'weight_3': 0.9253106338284529, 'weight_4': 0.4012297159067414, 'weight_5': 0.69

Best Optuna weights: {'social_media_ComplementNB_model': 0.03453101148662367, 'social_media_LinearSVC_(Optuna)_model': 0.033901522530360814, 'social_media_LogisticRegressionCV_model': 0.0443981063267201, 'wot_Logistic_Regression': 0.04312345731242262, 'wot_LinearSVC': 0.019294067762720704, 'wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True)': 0.067097186888403, 'wot_LinearSVC_+_Calibrated(isotonic,_ensemble=True)': 0.05589727813103788, 'dota_Logistic_Regression': 0.007315104199179971, 'dota_LinearSVC': 0.006711929930893352, 'dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False)': 0.009466695675019218, 'dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True)': 0.13002294395015693, 'combined_Logistic_Regression': 0.12907641439367634, 'combined_LinearSVC': 0.1378505003111681, 'combined_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True)': 0.12650144487217063, 'combined_LinearSVC_+_Calibrated(sigmoid,_ensemble=False)': 0.07140015474641971, 'dota_binary': 0.06545390453

In [23]:
pred_train = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=train[tc],
    weights=res[0]
)
metrics.metrics(train['label_binary'], pred_train)

Constructing confidence tensor...
Total models in ensemble: 18
Expected confidence shape: (n_samples=53707, n_classes=2)
Predicted labels shape: (53707,)


{'CV Macro F1': 0.967378039634579,
 'CV Weighted F1': 0.9763655124201833,
 'Accuracy': 0.9763904146573072,
 'Coverage': 1.0,
 'Precision': 0.9531496062992126,
 'FPR': 0.014537369591243373,
 'FNR': 0.05266864924088277,
 'TPR': 0.9473313507591172,
 'TNR': 0.9854626304087566}

In [24]:
pred_val = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=val[tc],
    weights=res[0]
)
metrics.metrics(val['label_binary'], pred_val)

Constructing confidence tensor...
Total models in ensemble: 18
Expected confidence shape: (n_samples=17056, n_classes=2)
Predicted labels shape: (17056,)


{'CV Macro F1': 0.8470311891529719,
 'CV Weighted F1': 0.8973321274940018,
 'Accuracy': 0.8987453095684803,
 'Coverage': 1.0,
 'Precision': 0.7880023296447292,
 'FPR': 0.054527750730282376,
 'FNR': 0.26963562753036435,
 'TPR': 0.7303643724696356,
 'TNR': 0.9454722492697176}

In [ ]:
thresholds = np.array([0.5, 0.9, 0.1])

res = social_gaming_ensemble.fit_weighted_confidence_majority_optuna_cv(
    X=train[tc],
    y=train['label_binary'],
    score_func=metrics.score,
    n_trials=100,
    random_state=CONFIG['seed'],
    thresholds=thresholds
)

Constructing confidence tensor...


In [ ]:
pred_train = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=train[tc],
    weights=res[0]
)
metrics.metrics(train['label_binary'], pred_train)

{'CV Macro F1': 0.8955727514433802,
 'CV Weighted F1': 0.9263699768114969,
 'Accuracy': 0.928631277114715,
 'Coverage': 1.0,
 'Precision': 0.9175613854915508,
 'FPR': 0.02157394512448386,
 'FNR': 0.23086555016434496,
 'TPR': 0.769134449835655,
 'TNR': 0.9784260548755161}

In [ ]:
pred_val = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=val[tc],
    weights=res[0]
)
metrics.metrics(val['label_binary'], pred_val)

{'CV Macro F1': 0.8258950745924274,
 'CV Weighted F1': 0.8858653276276821,
 'Accuracy': 0.8904784240150094,
 'Coverage': 1.0,
 'Precision': 0.809989875126561,
 'FPR': 0.04216912590817167,
 'FNR': 0.3522267206477733,
 'TPR': 0.6477732793522267,
 'TNR': 0.9578308740918283}

In [ ]:
res